07_formations_etl.ipynb — ETL for “Trenes despachados 2024”

Purpose: ingest the official formations dispatched dataset (formaciones_despachadas_2024.xlsx) and export a normalized 2024 table to data/processed/formaciones_2024.(csv|parquet) for use in the Dash KPIs.

In [1]:
# 1) Setup & Paths

from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)

def find_root(max_up=6):
    cur = Path.cwd()
    best = cur
    score_best = (-1, -1)
    for i in range(max_up + 1):
        up = cur if i == 0 else cur.parents[i-1]
        has_assets = (up / "assets").exists()
        has_data = (up / "data").exists()
        has_git = (up / ".git").exists()
        score = (int(has_assets and has_data), int(has_git))
        if score > score_best:
            best, score_best = up, score
    return best

ROOT = find_root()
RAW_DIR = ROOT / "data" / "raw" / "formaciones"
PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

INPUT_XLSX = RAW_DIR / "formaciones_despachadas_2024.xlsx"
OUTPUT_CSV = PROCESSED_DIR / "formaciones_2024.csv"
OUTPUT_PARQUET = PROCESSED_DIR / "formaciones_2024.parquet"
COMPARE_OUT = PROCESSED_DIR / "formations_compare_official_vs_prev.csv"

print("ROOT:", ROOT)
print("INPUT_XLSX:", INPUT_XLSX)
print("OUTPUT_CSV:", OUTPUT_CSV)
print("OUTPUT_PARQUET:", OUTPUT_PARQUET)


ROOT: c:\Users\do_ch\OneDrive\Escritorio\Proyectos\Proyectos GitHub\subte-dashboard
INPUT_XLSX: c:\Users\do_ch\OneDrive\Escritorio\Proyectos\Proyectos GitHub\subte-dashboard\data\raw\formaciones\formaciones_despachadas_2024.xlsx
OUTPUT_CSV: c:\Users\do_ch\OneDrive\Escritorio\Proyectos\Proyectos GitHub\subte-dashboard\data\processed\formaciones_2024.csv
OUTPUT_PARQUET: c:\Users\do_ch\OneDrive\Escritorio\Proyectos\Proyectos GitHub\subte-dashboard\data\processed\formaciones_2024.parquet


In [2]:
# 2) Inspect workbook (sheets & quick peek)

if not INPUT_XLSX.exists():
    raise FileNotFoundError(
        f"Expected Excel at {INPUT_XLSX}. "
        "Place the official file in data/raw/formaciones/ with that name."
    )

xls = pd.ExcelFile(INPUT_XLSX)
print("Sheets:", xls.sheet_names)

# Try to pick the most relevant sheet by a simple heuristic
preferred = [s for s in xls.sheet_names if "2024" in s.lower() or "despach" in s.lower() or "formac" in s.lower()]
sheet = preferred[0] if preferred else xls.sheet_names[0]
print("Selected sheet:", sheet)

peek = pd.read_excel(INPUT_XLSX, sheet_name=sheet, header=None, nrows=8)
display(peek)


Sheets: ['formaciones-despachadas-2024']
Selected sheet: formaciones-despachadas-2024


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,Fecha,Linea,Tipo,Regist,Orden,Tren,Nombre Formación,Modelo Formación,Causa A,Descripción A,Causa D,Descripción D,Cant Coches A,Cant Coches D,Km A,Km D,Tipo Viaje A,Tipo Viaje D,Hora Sale A,Hora Sale D
1,2024-01-01 00:00:00,A,F,1,1,3,228,CNR,,...,,...,0,5,0,5636,N,S,1900-01-01 00:00:00,1900-01-01 08:00:00
2,2024-01-01 00:00:00,A,F,2,2,4,217,CNR,,...,,...,0,5,0,8717,N,S,1900-01-01 00:00:00,1900-01-01 08:00:00
3,2024-01-01 00:00:00,A,F,3,3,5,213,CNR,,...,,...,0,5,0,9770,N,S,1900-01-01 00:00:00,1900-01-01 08:00:00
4,2024-01-01 00:00:00,A,F,4,4,6,214,CNR,,...,,...,5,5,1987,9770,S,S,1900-01-01 08:00:00,1900-01-01 08:09:20
5,2024-01-01 00:00:00,A,F,5,5,7,225,CNR,,...,,...,5,5,6264,9770,S,S,1900-01-01 08:00:00,1900-01-01 08:18:45
6,2024-01-01 00:00:00,A,F,6,6,1,227,CNR,,...,,...,5,5,9770,9770,S,S,1900-01-01 08:00:00,1900-01-01 08:26:30
7,2024-01-01 00:00:00,A,F,7,7,2,218,CNR,,...,,...,5,5,9770,9770,S,S,1900-01-01 08:07:38,1900-01-01 08:34:08


In [3]:
# 3) Flexible loader (column mapping)
# We try with header at row 0 first, then fallback by searching for a header row.

def norm(s: str) -> str:
    return str(s).strip().lower().replace(" ", "_")

def try_load_with_header(header_row: int):
    df = pd.read_excel(INPUT_XLSX, sheet_name=sheet, header=header_row)
    df.columns = [norm(c) for c in df.columns]
    return df

# attempt: header at row 0, else search within first 10 rows
df_candidates = []
try:
    df0 = try_load_with_header(0)
    df_candidates.append(df0)
except Exception as e:
    df_candidates = []

header_found = None
if df_candidates:
    header_found = 0
else:
    for h in range(1, 10):
        try:
            dfh = try_load_with_header(h)
            df_candidates.append(dfh)
            header_found = h
            break
        except Exception:
            continue

if not df_candidates:
    raise RuntimeError("Could not read a valid table from the Excel sheet with a guessed header row.")

raw = df_candidates[0].copy()
print(f"Header row used: {header_found}")
print("Columns detected:", list(raw.columns))

# Map columns
col_date = None
col_line = None
col_trains = None

date_candidates = [c for c in raw.columns if any(k in c for k in ["fecha", "date", "dia", "day"])]
line_candidates = [c for c in raw.columns if any(k in c for k in ["linea", "línea", "line"])]
trn_candidates  = [c for c in raw.columns if any(k in c for k in ["formac", "tren", "despach", "services", "servicios", "freq"])]

if date_candidates:
    col_date = date_candidates[0]
if line_candidates:
    col_line = line_candidates[0]
if trn_candidates:
    col_trains = trn_candidates[0]

print("Mapped columns →", {"date": col_date, "line": col_line, "trains": col_trains})

if not all([col_date, col_line, col_trains]):
    raise ValueError(
        "Could not map required columns. "
        f"Found: {list(raw.columns)}\n"
        "Need something like: fecha/date, linea/line, formaciones/trenes/servicios."
    )


Header row used: 0
Columns detected: ['fecha', 'linea', 'tipo', 'regist', 'orden', 'tren', 'nombre__formación', 'modelo__formación', 'causa_a', 'descripción_a', 'causa_d', 'descripción_d', 'cant_coches_a', 'cant_coches_d', 'km_a', 'km_d', 'tipo_viaje_a', 'tipo_viaje_d', 'hora_sale_a', 'hora_sale_d']
Mapped columns → {'date': 'fecha', 'line': 'linea', 'trains': 'tren'}


In [4]:
# 4) Normalize & Clean

df = raw[[col_date, col_line, col_trains]].copy()
df.columns = ["date", "line", "trains"]

# Parse date & numeric
df["date"] = pd.to_datetime(df["date"], errors="coerce", dayfirst=True)
df["trains"] = pd.to_numeric(df["trains"], errors="coerce")

# Normalize line labels to match the dashboard ("LineaA", "LineaB", ...)
def normalize_line(v: str) -> str:
    if pd.isna(v):
        return np.nan
    s = str(v).strip().replace(" ", "")
    # common inputs: "Linea A", "Línea A", "A", "LineaA"
    s = s.replace("Línea", "Linea").replace("línea", "Linea")
    if s.lower().startswith("linea"):
        return s
    # If it's a single letter (A/B/C/...), prefix
    if len(s) == 1 and s.isalpha():
        return f"Linea{s.upper()}"
    return s

df["line"] = df["line"].apply(normalize_line)

# Keep only 2024, drop empty rows
before = len(df)
df = df.dropna(subset=["date", "line", "trains"])
df = df[(df["date"].dt.year == 2024)]
after = len(df)

print(f"Dropped {before - after} rows due to NA or non-2024.")

# If multiple entries per day-line, aggregate
df = df.groupby(["date", "line"], as_index=False)["trains"].sum()

# Basic sanity checks
neg = (df["trains"] < 0).sum()
if neg > 0:
    print(f"WARNING: found {neg} negative values in 'trains'. Will set them to NaN.")
    df.loc[df["trains"] < 0, "trains"] = np.nan

df["trains"] = df["trains"].astype("Int64")
print("Final shape:", df.shape)
display(df.head(10))


Dropped 0 rows due to NA or non-2024.
Final shape: (2506, 3)


,date,line,trains
0,2024-01-01,LineaA,466
1,2024-01-01,LineaB,540
2,2024-01-01,LineaC,313
3,2024-01-01,LineaD,448
4,2024-01-01,LineaE,389
5,2024-01-01,LineaH,435
6,2024-01-01,LineaP,183
7,2024-01-02,LineaA,2123
8,2024-01-02,LineaB,1612
9,2024-01-02,LineaC,1273


In [5]:
# 5) Export & summary

df.to_csv(OUTPUT_CSV, index=False)
df.to_parquet(OUTPUT_PARQUET, index=False)

print("Saved:")
print(" -", OUTPUT_CSV)
print(" -", OUTPUT_PARQUET)

print("\nSummary by line (total trains in 2024):")
display(df.groupby("line")["trains"].sum().sort_values(ascending=False).to_frame("trains_total"))

print("\nMonthly totals by line:")
monthly = df.assign(year=df["date"].dt.year, month=df["date"].dt.month)
monthly = monthly.groupby(["year","month","line"], as_index=False)["trains"].sum()
display(monthly.head(12))


Saved:
 - c:\Users\do_ch\OneDrive\Escritorio\Proyectos\Proyectos GitHub\subte-dashboard\data\processed\formaciones_2024.csv
 - c:\Users\do_ch\OneDrive\Escritorio\Proyectos\Proyectos GitHub\subte-dashboard\data\processed\formaciones_2024.parquet

Summary by line (total trains in 2024):


,trains_total
line,
LineaA,781209
LineaB,605261
LineaH,500891
LineaC,473934
LineaE,438603
LineaD,375795
LineaP,124602



Monthly totals by line:


,year,month,line,trains
0,2024,1,LineaA,51972
1,2024,1,LineaB,42821
2,2024,1,LineaC,31863
3,2024,1,LineaD,8878
4,2024,1,LineaE,20056
5,2024,1,LineaH,27027
6,2024,1,LineaP,10453
7,2024,2,LineaA,49585
8,2024,2,LineaB,38332
9,2024,2,LineaC,30485


In [6]:
# 6) Optional comparison with previous processed file

prev_csv = PROCESSED_DIR / "formaciones_2024.csv"
prev_parq = PROCESSED_DIR / "formaciones_2024.parquet"

def load_prev() -> pd.DataFrame | None:
    # Try to load a previous version *other than the one we just wrote*
    # We'll try parquet first for quality, but skip if it's the same path we just wrote.
    try_paths = []
    if prev_parq.exists():
        try_paths.append(prev_parq)
    if prev_csv.exists():
        try_paths.append(prev_csv)
    # If both exist and are identical to the outputs we just saved, we'll just skip compare.
    if not try_paths:
        return None
    # Load whichever exists; if it's just created, it's equal → comparison is trivial.
    p = try_paths[0]
    dfp = pd.read_parquet(p) if p.suffix == ".parquet" else pd.read_csv(p, parse_dates=["date"])
    dfp["date"] = pd.to_datetime(dfp["date"], errors="coerce")
    dfp["trains"] = pd.to_numeric(dfp["trains"], errors="coerce")
    dfp["line"] = dfp["line"].astype(str)
    return dfp

prev = load_prev()

if prev is None:
    print("No previous formations file found for comparison (this is fine).")
else:
    # Build monthly totals for both
    cur_m = df.assign(ym=df["date"].dt.to_period("M")).groupby(["ym","line"], as_index=False)["trains"].sum()
    prv_m = prev.assign(ym=prev["date"].dt.to_period("M")).groupby(["ym","line"], as_index=False)["trains"].sum()
    cmp = pd.merge(cur_m, prv_m, on=["ym","line"], how="outer", suffixes=("_official","_prev"))
    cmp["diff"] = cmp["trains_official"].fillna(0) - cmp["trains_prev"].fillna(0)
    cmp = cmp.sort_values(["ym","line"])
    cmp["ym"] = cmp["ym"].astype(str)

    cmp.to_csv(COMPARE_OUT, index=False)
    print("Comparison exported →", COMPARE_OUT)
    display(cmp.head(20))


Comparison exported → c:\Users\do_ch\OneDrive\Escritorio\Proyectos\Proyectos GitHub\subte-dashboard\data\processed\formations_compare_official_vs_prev.csv


,ym,line,trains_official,trains_prev,diff
0,2024-01,LineaA,51972,51972,0
1,2024-01,LineaB,42821,42821,0
2,2024-01,LineaC,31863,31863,0
3,2024-01,LineaD,8878,8878,0
4,2024-01,LineaE,20056,20056,0
5,2024-01,LineaH,27027,27027,0
6,2024-01,LineaP,10453,10453,0
7,2024-02,LineaA,49585,49585,0
8,2024-02,LineaB,38332,38332,0
9,2024-02,LineaC,30485,30485,0
